<a href="https://colab.research.google.com/github/Mohamed-Heni25/fraud-detection-job-offers/blob/main/nvda_pred_23/11/26_to_07/01/27_SentimentVsTsForcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q finnhub-python yfinance transformers chronos-forecasting xgboost scikit-learn torch pandas pytz


Mounted at /content/drive


In [7]:
import finnhub
import pandas as pd
import time
from datetime import datetime, timedelta

FINNHUB_KEY = "dc28p6pr01qjj5fhmkj0dc28p6pr01qjj5fhmkjg"
finnhub_client = finnhub.Client(api_key=FINNHUB_KEY)

end_date = datetime.now().date()
start_date = end_date - timedelta(days=365)

all_articles = []
curr_start = start_date

while curr_start < end_date:
    curr_end = min(curr_start + timedelta(days=7), end_date)
    news = finnhub_client.company_news(
        "NVDA",
        _from=curr_start.strftime("%Y-%m-%d"),
        to=curr_end.strftime("%Y-%m-%d")
    )
    all_articles.extend(news)
    curr_start = curr_end + timedelta(days=1)
    time.sleep(1.1)  # Stay safely under Finnhub's 60 calls/min limit

news_df = pd.DataFrame(all_articles).drop_duplicates(subset=["id"])

# Keep headlines specifically focused on NVIDIA
mask = news_df["headline"].str.contains("NVIDIA|NVDA", case=False, na=False)
news_df = news_df[mask].copy()

# Convert UTC timestamps to US Eastern Time & shift after-close news (>= 4:00 PM ET)
dt_et = pd.to_datetime(news_df["datetime"], unit="s", utc=True).dt.tz_convert("America/New_York")
after_close = dt_et.dt.hour >= 16
dt_et = dt_et.where(~after_close, dt_et + pd.Timedelta(days=1))

# Roll Saturday (5) -> Monday (+2 days) and Sunday (6) -> Monday (+1 day)
dow = dt_et.dt.dayofweek
dt_et = dt_et.where(dow != 5, dt_et + pd.Timedelta(days=2))
dt_et = dt_et.where(dow != 6, dt_et + pd.Timedelta(days=1))

news_df["trade_date"] = dt_et.dt.date
print(f"Cleaned & aligned {len(news_df)} NVDA-specific headlines.")


Cleaned & aligned 2912 NVDA-specific headlines.


In [8]:
import torch
import yfinance as yf
from transformers import pipeline

sentiment_pipe = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

headlines = news_df["headline"].fillna("").tolist()
batch_results = sentiment_pipe(headlines, batch_size=32)

news_df["net_sentiment"] = [
    {item["label"]: item["score"] for item in res}["positive"] -
    {item["label"]: item["score"] for item in res}["negative"]
    for res in batch_results
]

daily_sentiment = news_df.groupby("trade_date").agg(
    sentiment_mean=("net_sentiment", "mean"),
    sentiment_std=("net_sentiment", "std"),
    article_count=("id", "count")
).reset_index()
daily_sentiment["trade_date"] = pd.to_datetime(daily_sentiment["trade_date"])

# Fetch matching NVDA daily prices
nvda = yf.download("NVDA", start=start_date, end=end_date, interval="1d", progress=False)
price_df = nvda[["Close", "Volume"]].reset_index()
price_df.columns = ["timestamp", "close", "volume"]
price_df["timestamp"] = pd.to_datetime(price_df["timestamp"]).dt.tz_localize(None)

df = pd.merge(price_df, daily_sentiment, left_on="timestamp", right_on="trade_date", how="left").drop(columns=["trade_date"])
df[["sentiment_mean", "sentiment_std", "article_count"]] = df[["sentiment_mean", "sentiment_std", "article_count"]].fillna(0.0)

# Save permanently to Google Drive
SAVE_PATH = "/content/drive/MyDrive/nvda_price_and_sentiment.csv"
df.to_csv(SAVE_PATH, index=False)
print(f"Saved dataset ({len(df)} trading days) to {SAVE_PATH}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_445/3425640941.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nvda = yf.download("NVDA", start=start_date, end=end_date, interval="1d", progress=False)


Saved dataset (250 trading days) to /content/drive/MyDrive/nvda_price_and_sentiment.csv


In [13]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("/content/drive/MyDrive/nvda_price_and_sentiment.csv")

# Engineer stationary features (using past data only—no leakage)
df["ret_1d"] = df["close"].pct_change(1)
df["ret_5d"] = df["close"].pct_change(5)
df["vol_10d"] = df["ret_1d"].rolling(10).std()
df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean()
df["sent_3d_avg"] = df["sentiment_mean"].rolling(3).mean()

# Target: Next trading day's percentage return
df["target_next_ret"] = df["close"].pct_change(1).shift(-1)
df = df.dropna().reset_index(drop=True)

price_features = ["ret_1d", "ret_5d", "vol_10d", "vol_ratio"]
hybrid_features = price_features + ["sentiment_mean", "sentiment_std", "article_count", "sent_3d_avg"]

# Walk-forward validation across the final 60 trading days
test_days = 60
preds_price, preds_hybrid, actuals = [], [], []

for i in range(len(df) - test_days, len(df)):
    train = df.iloc[:i]
    test = df.iloc[i:i+1]

    m_price = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    m_price.fit(train[price_features], train["target_next_ret"])
    preds_price.append(m_price.predict(test[price_features])[0])

    m_hybrid = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    m_hybrid.fit(train[hybrid_features], train["target_next_ret"])
    preds_hybrid.append(m_hybrid.predict(test[hybrid_features])[0])

    actuals.append(test["target_next_ret"].values[0])

print(f"Price-Only MAE:       {mean_absolute_error(actuals, preds_price):.5f}")
print(f"Price + Sentiment MAE:{mean_absolute_error(actuals, preds_hybrid):.5f}")


Price-Only MAE:       0.03003
Price + Sentiment MAE:0.02880


In [19]:
import pandas as pd
import torch
from chronos import BaseChronosPipeline

raw_df = pd.read_csv("/content/drive/MyDrive/nvda_price_and_sentiment.csv")
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"])
raw_df = raw_df.sort_values("timestamp").drop_duplicates(subset=["timestamp"])

# 1. Reindex onto a strict Monday–Friday business-day grid ("B") so Chronos sees uniform spacing
full_b_days = pd.date_range(start=raw_df["timestamp"].min(), end=raw_df["timestamp"].max(), freq="B")
chronos_df = raw_df.set_index("timestamp").reindex(full_b_days)

# Forward-fill closing prices on market holidays; set holiday news/sentiment to 0
chronos_df["close"] = chronos_df["close"].ffill()
chronos_df["sentiment_mean"] = chronos_df["sentiment_mean"].fillna(0.0)
chronos_df["article_count"] = chronos_df["article_count"].fillna(0.0)

chronos_df = chronos_df[["close", "sentiment_mean", "article_count"]].reset_index()
chronos_df = chronos_df.rename(columns={"index": "timestamp", "close": "target"})
chronos_df["item_id"] = "NVDA"

# 2. Load Chronos-2 & pass freq="B" explicitly
chronos_pipe = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda" if torch.cuda.is_available() else "cpu"
)

forecast = chronos_pipe.predict_df(
    chronos_df,
    prediction_length=30,
    quantile_levels=[0.3, 0.5, 0.6],
    id_column="item_id",
    timestamp_column="timestamp",
    target="target",
    freq="B",
)

print(forecast[["timestamp", "0.3", "0.5", "0.6"]])


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

    timestamp         0.3         0.5         0.6
0  2026-11-24  154.674255  161.103149  164.160614
1  2026-11-25  153.558380  161.481369  165.188904
2  2026-11-26  152.073242  161.399918  165.475586
3  2026-11-27  151.826950  161.614197  166.407669
4  2026-11-30  151.007507  161.659424  166.406250
5  2026-12-01  150.584457  161.377136  166.646530
6  2026-12-02  150.167465  161.581085  167.250397
7  2026-12-03  149.979462  162.041962  167.787781
8  2026-12-04  149.698334  162.151581  168.030380
9  2026-12-07  149.209290  162.371902  168.604095
10 2026-12-08  148.827789  161.925461  168.095932
11 2026-12-09  148.509430  161.992889  168.350220
12 2026-12-10  148.361389  162.377884  168.984741
13 2026-12-11  148.118073  162.720825  169.463318
14 2026-12-14  148.387695  163.233307  170.063568
15 2026-12-15  148.906494  163.447693  170.534729
16 2026-12-16  147.713654  163.159546  170.600220
17 2026-12-17  148.562393  163.813873  171.119812
18 2026-12-18  147.070770  162.798645  170.307739
